<a href="https://colab.research.google.com/github/en970/gausscapture/blob/main/notebooks/GaussCapture_Colab_Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GaussCapture · Colab trainer

Trains a 3D Gaussian splat from a `dataset.zip` produced by
`gausscapture colab <project>`.

**Runtime → Change runtime type → GPU.** A T4 is enough.

---

### Why gsplat and not the reference implementation

The original 3D Gaussian Splatting code from Inria is licensed for
**non-commercial research only**. Anything trained with it inherits that
restriction, which would make GaussCapture's MIT licence a promise it cannot
keep. [gsplat](https://github.com/nerfstudio-project/gsplat) is Apache-2.0,
actively maintained, and produces equivalent quality, so it is what this
notebook uses. See `docs/DEPENDENCIES.md` in the repository.


In [1]:
# Confirm a GPU is attached. Without one the rest of this notebook will not run.
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU.'
print(f'torch {torch.__version__}  cuda {torch.version.cuda}  {torch.cuda.get_device_name(0)}')


Wed Jul 29 19:41:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1 · Install gsplat

Installed from PyPI, which compiles its CUDA kernels **on first use** rather than
at install time. That matters here: building at install time is what fails when
Colab's torch and CUDA are newer than any prebuilt wheel, and prebuilt wheels also
require installing gsplat's dependencies by hand.

The last step deliberately triggers that first compile. It takes a few minutes and
it is better spent here, where a failure is obvious, than twenty minutes into
training.


In [ ]:
!pip install -q ninja numpy jaxtyping rich
!pip install -q gsplat

# simple_trainer.py needs a few extras beyond the library itself.
!pip install -q tyro tqdm 'imageio[ffmpeg]' torchmetrics scikit-learn opencv-python-headless plyfile

import torch, gsplat
print('gsplat', gsplat.__version__, '· torch', torch.__version__, '· cuda', torch.version.cuda)

# Force the JIT build now, with a rasterisation small enough to be instant once
# compiled. If the toolchain is broken this is where it says so.
print('\ncompiling CUDA kernels (first run only, a few minutes)...')
means = torch.rand(16, 3, device='cuda')
quats = torch.nn.functional.normalize(torch.rand(16, 4, device='cuda'), dim=-1)
scales = torch.rand(16, 3, device='cuda') * 0.1
opacities = torch.rand(16, device='cuda')
colors = torch.rand(16, 3, device='cuda')
viewmats = torch.eye(4, device='cuda')[None]
Ks = torch.tensor([[[100.0, 0, 32], [0, 100.0, 32], [0, 0, 1]]], device='cuda')
render, alpha, _ = gsplat.rasterization(
    means, quats, scales, opacities, colors, viewmats, Ks, width=64, height=64)
print('kernels ready · test render', tuple(render.shape))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 59.8 MB/s eta 0:00:00


## 2 · Upload the dataset

Upload the `dataset.zip` that `gausscapture colab` produced. It already contains
`images/` and `sparse/0/` in the layout every trainer expects, so nothing here
needs converting.


In [ ]:
import pathlib, shutil, zipfile
from google.colab import files

uploaded = files.upload()
archive = next(iter(uploaded))

data = pathlib.Path('/content/dataset')
if data.exists():
    shutil.rmtree(data)
data.mkdir(parents=True)
with zipfile.ZipFile(archive) as z:
    z.extractall(data)

images = sorted((data / 'images').glob('*.jpg'))
sparse = data / 'sparse' / '0'
assert images, 'No images/ in the archive.'
assert sparse.exists(), 'No sparse/0/ in the archive.'
print(f'{len(images)} images, model files: {sorted(p.name for p in sparse.iterdir())}')


## 3 · Train

`mcmc` is gsplat's densification strategy and is the better default for phone
captures, which have uneven coverage. `--use-bilateral-grid` compensates for
residual exposure drift between frames; harmless when exposure was locked, and
worth several dB when it was not.

30,000 steps is the standard budget and takes roughly 20-40 minutes on a T4. Drop
to 7,000 for a quick look — it reaches most of the quality.


In [ ]:
STEPS = 30_000          # 7_000 for a fast preview
DOWNSCALE = 1           # raise to 2 if the GPU runs out of memory

import pathlib
trainer = pathlib.Path('/content/gsplat_examples/simple_trainer.py')
if not trainer.exists():
    !git clone -q --depth 1 https://github.com/nerfstudio-project/gsplat.git /content/gsplat_src
    !cp -r /content/gsplat_src/examples /content/gsplat_examples

result = pathlib.Path('/content/result')
!python {trainer} mcmc \
    --data_dir /content/dataset \
    --data_factor {DOWNSCALE} \
    --result_dir {result} \
    --max_steps {STEPS} \
    --save_ply \
    --disable_viewer \
    --use_bilateral_grid


## 4 · Collect the splat

The `.ply` is the trained Gaussian splat. Drop it into
[SuperSplat](https://superspl.at/editor) or [Spark](https://sparkjs.dev) to view it,
or import it back into GaussCapture with `gausscapture` to preview and export.


In [ ]:
import pathlib, shutil

plys = sorted(pathlib.Path('/content/result').rglob('*.ply'), key=lambda p: p.stat().st_size)
assert plys, 'Training produced no .ply — check the log above.'
splat = plys[-1]                      # the largest is the final model
print(f'{splat}  ({splat.stat().st_size/1e6:.1f} MB)')

bundle = shutil.make_archive('/content/gausscapture_splat', 'zip', splat.parent)
from google.colab import files
files.download(bundle)
